# Reproduction - Deep Transformer Models for Time Series Forecasting: The Influenza Prevalence Case

State-level Transformer reproduction with an added classical ARIMA baseline.

## Load Dataset

In [1]:
from dataset import Dataset

dataset = Dataset(path="./data/state/ILINet.csv")
train_loader, val_loader, test_loader= dataset.get_train_val_test_loader(
    history=10,
    future=4,
    batch_size=64
)

Train samples: 15295
Validation samples: 1657
Test samples: 4332


In [4]:
import warnings
import pandas as pd
from sklearn.metrics import mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import pearsonr
import random
import numpy as np
import matplotlib.pyplot as plt
import copy

try:
    from statsmodels.tsa.arima.model import ARIMA
except ImportError:
    # Uncomment the next two lines if statsmodels is missing in your environment.
    # import sys
    # !{sys.executable} -m pip install -q statsmodels
    raise ImportError("statsmodels is required for the ARIMA baseline. Install it with: pip install statsmodels")


HISTORY = 10
FUTURE = 4
TEST_SIZE = 0.2
VAL_SIZE = 0.1
ARIMA_ORDER = (2, 0, 0)

# Use a small integer such as 20 while debugging. Use None for the full final run.
MAX_TEST_WINDOWS_PER_STATE = None


def safe_pearson(y_true, y_pred):
    """Pearson correlation with a safe fallback for nearly constant arrays."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    if len(y_true) < 2 or np.std(y_true) < 1e-8 or np.std(y_pred) < 1e-8:
        return np.nan
    return pearsonr(y_true, y_pred)[0]


def get_state_test_windows(dataset, history=10, future=4, test_size=0.2, val_size=0.1):
    """Build test windows for each state using the same split logic as dataset.py."""
    df = dataset.clean_dataframe()

    state_windows = {}
    for state, g in df.groupby("REGION"):
        g = g.sort_values(["YEAR", "WEEK"])
        series = g["%UNWEIGHTED ILI"].values.astype(np.float32)

        if len(series) < history + future:
            continue

        _, _, _, _, x_test, y_test = dataset.build_split_windows(
            series=series,
            history=history,
            future=future,
            test_size=test_size,
            val_size=val_size,
        )

        if len(x_test) > 0:
            state_windows[state] = (x_test, y_test)

    return state_windows


def arima_forecast_window(history_values, future=4, order=(2, 0, 0)):
    """Fit ARIMA on one history window and forecast the next future steps."""
    history_values = np.asarray(history_values, dtype=np.float32)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = ARIMA(history_values, order=order)
            fitted = model.fit()
            forecast = fitted.forecast(steps=future)
        return np.asarray(forecast, dtype=np.float32)
    except Exception:
        # Conservative fallback: repeat the last observed value if ARIMA fails to converge.
        return np.repeat(history_values[-1], future).astype(np.float32)


def arima_forecast_state(x_windows, future=4, order=(2, 0, 0), max_windows=None):
    """Run rolling-window ARIMA forecasts for one state."""
    if max_windows is not None:
        x_windows = x_windows[:max_windows]

    preds = []
    for x in x_windows:
        preds.append(arima_forecast_window(x, future=future, order=order))

    return np.stack(preds, axis=0)


def summarize_forecasts(y_true, y_pred, label):
    """Return overall and horizon-wise RMSE, MAE, and Pearson metrics."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rows = []
    rows.append({
        "Model": label,
        "Horizon": "Overall",
        "RMSE": np.sqrt(mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1))),
        "MAE": mean_absolute_error(y_true.reshape(-1), y_pred.reshape(-1)),
        "Pearson": safe_pearson(y_true.reshape(-1), y_pred.reshape(-1)),
    })

    for h in range(y_true.shape[1]):
        rows.append({
            "Model": label,
            "Horizon": f"Week+{h + 1}",
            "RMSE": np.sqrt(mean_squared_error(y_true[:, h], y_pred[:, h])),
            "MAE": mean_absolute_error(y_true[:, h], y_pred[:, h]),
            "Pearson": safe_pearson(y_true[:, h], y_pred[:, h]),
        })

    return pd.DataFrame(rows)


In [5]:
state_windows = get_state_test_windows(
    dataset,
    history=HISTORY,
    future=FUTURE,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
)

print(f"Number of states/regions evaluated: {len(state_windows)}")

all_arima_preds = []
all_targets = []
state_metric_rows = []

for state, (x_test_state, y_test_state) in state_windows.items():
    if MAX_TEST_WINDOWS_PER_STATE is not None:
        x_eval = x_test_state[:MAX_TEST_WINDOWS_PER_STATE]
        y_eval = y_test_state[:MAX_TEST_WINDOWS_PER_STATE]
    else:
        x_eval = x_test_state
        y_eval = y_test_state

    arima_pred = arima_forecast_state(
        x_eval,
        future=FUTURE,
        order=ARIMA_ORDER,
        max_windows=None,
    )

    all_arima_preds.append(arima_pred)
    all_targets.append(y_eval)

    state_metric_rows.append({
        "State": state,
        "ARIMA_RMSE": np.sqrt(mean_squared_error(y_eval.reshape(-1), arima_pred.reshape(-1))),
        "ARIMA_MAE": mean_absolute_error(y_eval.reshape(-1), arima_pred.reshape(-1)),
        "ARIMA_Pearson": safe_pearson(y_eval.reshape(-1), arima_pred.reshape(-1)),
        "Num_Test_Windows": len(y_eval),
    })

arima_preds = np.concatenate(all_arima_preds, axis=0)
targets = np.concatenate(all_targets, axis=0)

arima_metrics = summarize_forecasts(targets, arima_preds, "State ARIMA")
state_metrics = pd.DataFrame(state_metric_rows).sort_values("ARIMA_RMSE")

print("Aggregated state-level ARIMA metrics:")
display(arima_metrics)

print("Per-state ARIMA metrics:")
display(state_metrics)


Number of states/regions evaluated: 54


NameError: name 'mean_squared_error' is not defined

In [ ]:
# Save ARIMA baseline outputs for comparison with Transformer and LSTM results.
arima_metrics.to_csv("state_arima_metrics.csv", index=False)
state_metrics.to_csv("state_arima_metrics_by_state.csv", index=False)
np.save("state_arima_preds.npy", arima_preds)
np.save("state_arima_targets.npy", targets)

print("Saved files:")
print("- state_arima_metrics.csv")
print("- state_arima_metrics_by_state.csv")
print("- state_arima_preds.npy")
print("- state_arima_targets.npy")


In [ ]:
# Plot overall and horizon-wise RMSE for state-level ARIMA.
plt.figure(figsize=(7, 5))
plt.bar(arima_metrics["Horizon"], arima_metrics["RMSE"])
plt.ylabel("RMSE")
plt.title("State-Level ARIMA RMSE by Forecast Horizon")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("state_arima_rmse_by_horizon.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(arima_metrics["Horizon"], arima_metrics["MAE"])
plt.ylabel("MAE")
plt.title("State-Level ARIMA MAE by Forecast Horizon")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("state_arima_mae_by_horizon.png", dpi=200)
plt.show()


In [ ]:
# Plot example forecast windows from the aggregated state-level test set.
num_examples = min(4, len(targets))
example_indices = np.linspace(0, len(targets) - 1, num_examples, dtype=int)

for idx in example_indices:
    plt.figure(figsize=(7, 4))
    horizon = np.arange(1, FUTURE + 1)
    plt.plot(horizon, targets[idx], marker="o", label="Actual")
    plt.plot(horizon, arima_preds[idx], marker="o", label="ARIMA")
    plt.xlabel("Forecast Horizon")
    plt.ylabel("% Unweighted ILI")
    plt.title(f"State-Level ARIMA Forecast Example #{idx}")
    plt.xticks(horizon, [f"Week+{h}" for h in horizon])
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Improvement - Deep Transformer Models for Time Series Forecasting: The Influenza Prevalence Case